# Tenacious-Bench v0.1 — ORPO Judge Training

**Path B:** Preference-tuned constraint judge via ORPO (Hong et al., EMNLP 2024)  
**Backbone:** Qwen 3.5 0.5B-Instruct (or 1.5B if VRAM allows)  
**Framework:** Unsloth + TRL ORPOTrainer  
**Target:** Free Colab T4 (16 GB VRAM)

In [1]:
!nvidia-smi

Sat May  2 05:29:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%capture

!pip install unsloth
!pip install --upgrade trl peft accelerate datasets huggingface_hub bitsandbytes

In [5]:
# Cell 2 — Imports and reproducibility seed
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from huggingface_hub import login
from transformers import TrainingArguments
from trl.experimental.orpo import ORPOConfig, ORPOTrainer
from unsloth import FastLanguageModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

/tmp/ipykernel_5062/173731043.py:13: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch  : 2.10.0+cu128
CUDA     : True — Tesla T4
VRAM     : 15.6 GB


In [6]:
# Cell 3 — Config (edit here, nowhere else)
MODEL_NAME   = "Qwen/Qwen3.5-0.8B"   
MAX_SEQ_LEN  = 512
DTYPE        = None        # Unsloth picks float16 on T4 automatically
LOAD_IN_4BIT = False
LORA_RANK    = 8
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05
ORPO_BETA    = 0.1
LR           = 2e-4
EPOCHS       = 3
BATCH_SIZE   = 1
GRAD_ACCUM   = 4           # effective batch = 4
HF_REPO      = "marthaket/tenacious-judge-orpo-qwen"
DATA_FILE    = "orpo_pairs.jsonl"

print(f"Model : {MODEL_NAME}")
print(f"Repo  : {HF_REPO}")


Model : Qwen/Qwen3.5-0.8B
Repo  : marthaket/tenacious-judge-orpo-qwen


In [7]:
# Cell 4 — HuggingFace login
# Set HF_TOKEN in Colab Secrets (key icon in sidebar) rather than pasting here.
hf_token = os.environ.get("HF_TOKEN") or input("Paste HF write token: ").strip()
login(token=hf_token, add_to_git_credential=False)
print("Logged in to HuggingFace.")

Logged in to HuggingFace.


In [8]:
# Cell 5 — Load model + tokenizer via Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print(f"Loaded {MODEL_NAME}")
print(f"VRAM used: {torch.cuda.memory_allocated() // 1024**2} MB")


==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Loaded Qwen/Qwen3.5-0.8B
VRAM used: 1644 MB


In [9]:
# Cell 6 — Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


trainable params: 3,194,880 || all params: 856,180,800 || trainable%: 0.3732


In [10]:
# Cell 7 — Load dataset
raw = [json.loads(l) for l in Path(DATA_FILE).read_text().splitlines() if l.strip()]
print(f"Loaded {len(raw)} pairs")

# ORPOTrainer expects: prompt (list of messages), chosen (list), rejected (list)
# The orpo_pairs.jsonl already has this format.

# Shuffle with fixed seed before split
random.shuffle(raw)
split = int(0.9 * len(raw))
train_data = raw[:split]
eval_data  = raw[split:]
print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")

train_ds = Dataset.from_list(train_data)
eval_ds  = Dataset.from_list(eval_data)

# Sanity check one pair
ex = train_data[0]
print("\nExample prompt[1] (user turn, first 200 chars):")
print(ex["prompt"][1]["content"][:200])
print("\nExample chosen (first 150 chars):")
print(ex["chosen"][0]["content"][:150])
print("\nExample rejected (first 150 chars):")
print(ex["rejected"][0]["content"][:150])

Loaded 123 pairs
Train: 110 | Eval: 13

Example prompt[1] (user turn, first 200 chars):
Prospect: Atommap Corp ()
Employees: 100
Funding: no funding data
Signal: 0 open roles, velocity=insufficient_signal, confidence=0.00, ai_maturity=0/3
Segment: abstain

Available engineering capacity:

Example chosen (first 150 chars):
Hi there,

I noticed that Atommap Corp is focused on engineering initiatives, and I wanted to reach out to see if there might be any upcoming projects

Example rejected (first 150 chars):
Hi — Atommap Corp is on a serious hiring tear right now. With your aggressive expansion plans, Tenacious can help you scale your engineering team 3-5x


In [17]:
# Cell 8 — ORPO training config
orpo_config = ORPOConfig(
    beta=ORPO_BETA,
    max_length=MAX_SEQ_LEN,
    # max_prompt_length=MAX_SEQ_LEN // 2,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="no",
    eval_steps=25,
    save_strategy="steps",
    save_steps=50,
    output_dir="./orpo_checkpoints",
    seed=SEED,
    report_to="none",    # swap to "wandb" if you have it configured
)
print("ORPOConfig ready.")
print(f"  Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Precision            : {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


ORPOConfig ready.
  Effective batch size : 4
  Precision            : fp16


In [18]:
# Cell 9 — Trainer init
trainer = ORPOTrainer(
    model=model,
    args=orpo_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer.tokenizer,
)
print("ORPOTrainer ready.")
print(f"Training steps: {len(trainer.get_train_dataloader()) * EPOCHS}")

Map:   0%|          | 0/110 [00:00<?, ? examples/s]

Map:   0%|          | 0/110 [00:00<?, ? examples/s]

Map:   0%|          | 0/110 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

ORPOTrainer ready.
Training steps: 330


In [19]:
# Cell 10 — Train
# Wall time ~30–60 min on T4 for 3 epochs over 120 pairs.
# If loss is not decreasing by step 50, check orpo_pairs.jsonl formatting.
# If OOM: reduce BATCH_SIZE to 1 or switch MODEL_NAME to 0.5B.
trainer_stats = trainer.train()

print(f"\nTraining complete.")
print(f"  Runtime      : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Final loss   : {trainer_stats.metrics['train_loss']:.4f}")
print(f"  Samples/sec  : {trainer_stats.metrics['train_samples_per_second']:.2f}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 110 | Num Epochs = 3 | Total steps = 84
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 3,194,880 of 856,180,800 (0.37% trained)


Step,Training Loss
10,0.877501
20,0.579603
30,0.361121
40,0.265531
50,0.308050
60,0.200631
70,0.187693
80,0.192032


Unsloth: Restored added_tokens_decoder metadata in ./orpo_checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./orpo_checkpoints/checkpoint-84/tokenizer_config.json.



Training complete.
  Runtime      : 659s
  Final loss   : 0.3606
  Samples/sec  : 0.50


In [20]:
# Cell 11 — Log loss curve to file (for training_run.log)
import csv

log_history = trainer.state.log_history
loss_rows = [
    {"step": e["step"], "loss": e.get("loss", ""), "eval_loss": e.get("eval_loss", "")}
    for e in log_history if "step" in e
]

with open("training_run.log", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["step", "loss", "eval_loss"])
    writer.writeheader()
    writer.writerows(loss_rows)

print(f"Loss curve written to training_run.log ({len(loss_rows)} rows)")
print("Download this file — commit it to training/ in your repo.")

# Also write hyperparams
hparams = {
    "model": MODEL_NAME,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "orpo_beta": ORPO_BETA,
    "lr": LR,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "max_seq_len": MAX_SEQ_LEN,
    "seed": SEED,
    "train_pairs": len(train_data),
    "eval_pairs": len(eval_data),
    "final_train_loss": trainer_stats.metrics["train_loss"],
    "runtime_seconds": trainer_stats.metrics["train_runtime"],
}
with open("hyperparams.json", "w") as f:
    json.dump(hparams, f, indent=2)
print("Hyperparameters written to hyperparams.json")

Loss curve written to training_run.log (9 rows)
Download this file — commit it to training/ in your repo.
Hyperparameters written to hyperparams.json


In [21]:
# Cell 12 — Quick sanity inference (before pushing)
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_prompt = [
    {"role": "system", "content": "You are a B2B sales rep for Tenacious. Never say 'bench' to prospects."},
    {"role": "user",   "content": "Prospect: Atommap Corp | 0 open roles | confidence=0.0 | rust=0 available.\n\nWrite a compliant cold outreach email."},
]

text = tokenizer.apply_chat_template(
    test_prompt, tokenize=False, add_generation_prompt=True
)
# Pass text= explicitly — avoids positional-arg collision in newer transformers
inputs = tokenizer(text=text, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, use_cache=True)

generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Model output:")
print(generated)
assert "bench" not in generated.lower() or "benchmark" in generated.lower(), "FAIL"
print("Sanity check passed.")


Model output:
Hi [Prospect's Name],

I noticed that Atommap Corp is currently looking to fill a few roles, and I wanted to reach out to see if you might be open to discussing your needs. At Tenacious, we specialize in building robust engineering teams that can handle complex projects efficiently.

While I don’t currently have a specific role for you, I’d love to learn more about your current challenges and how we might be able to support your team. Would you be open to a brief conversation? You can schedule a time that works for you using my calendar link: [Insert Your Calendar Link].

Looking forward to connecting!

Best,  
[Your Name]  
[Your Position]  
Tenacious
Sanity check passed.


In [22]:
# Cell 13 — Push LoRA adapter to HuggingFace Hub
# Pushes adapter weights only — NOT the merged backbone.
model.push_to_hub(HF_REPO, token=hf_token)
tokenizer.push_to_hub(HF_REPO, token=hf_token)
print(f"Adapter pushed to: https://huggingface.co/{HF_REPO}")
print("Backbone NOT uploaded — adapter only, per project spec.")

README.md:   0%|          | 0.00/529 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|4         |  559kB / 12.8MB            

Saved model to https://huggingface.co/marthaket/tenacious-judge-orpo-qwen


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpf_jk7mj0/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpf_jk7mj0/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

Adapter pushed to: https://huggingface.co/marthaket/tenacious-judge-orpo-qwen
Backbone NOT uploaded — adapter only, per project spec.


In [23]:
# Cell 14 — Download files needed for the repo
# After this cell, use File → Download in the Colab sidebar to get:
#   training_run.log  → commit to training/
#   hyperparams.json  → commit to training/
from google.colab import files
files.download("training_run.log")
files.download("hyperparams.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>